# Algoritmo Genético para o CVRP

**Disciplina:** Inteligência Artificial e Aprendizado de Máquina — 2026/1  
**Professor:** Gabriel de Oliveira Ramos  
**Grupo 8:** Erik Morbach, Gabriel Farias, Lucas Escopelli

---

## Resumo

Este trabalho apresenta a implementação de um Algoritmo Genético (AG) para o *Capacitated Vehicle Routing Problem* (CVRP). O CVRP é um problema NP-difícil: dado um depósito e um conjunto de cidades com demandas individuais, deve-se determinar rotas para uma frota de veículos com capacidade limitada, minimizando (1) o número de veículos e (2) a distância total percorrida.

Implementamos e comparamos **sete operadores genéticos**: dois de seleção (Torneio k=3 e k=7), três de cruzamento (OX, PMX e ER-Q) e dois de mutação (Swap e 2-opt). O operador ER-Q (*Edge Quality Aware Crossover*) representa o estado da arte em cruzamentos para CVRP, sendo o 7º operador exigido pelo enunciado (Chitty, Yates e Keedwell, GECCO '22). A população é inicializada com uma heurística de pivôs geográficos. Os experimentos são executados nas instâncias `eil33`, `eil51` e `eil76`.

**Link do vídeo:** *(a preencher antes da entrega)*

## 1. Descrição do Problema

O **Capacitated Vehicle Routing Problem (CVRP)** consiste em, dado:
- Um **depósito** central (ponto de partida e chegada de todos os veículos),
- Um conjunto de **cidades** com coordenadas $(x, y)$ e demanda $d_i$,
- Uma **capacidade máxima** $Q$ para cada veículo,

encontrar um conjunto de rotas tal que:
- Cada cidade seja visitada **exatamente uma vez**;
- A soma das demandas de cada rota **não ultrapasse** $Q$;
- O **número de veículos** seja mínimo;
- A **distância total** percorrida seja mínima (objetivo secundário).

O problema é **NP-difícil**, o que torna inviável a busca exaustiva para instâncias com mais de ~20 cidades, justificando o uso de meta-heurísticas como Algoritmos Genéticos.

## 2. Leitura e Visualização dos Dados

As instâncias estão no formato `.vrp` (TSPLIB).

In [ ]:
import math
import random
import matplotlib.pyplot as plt
import matplotlib.cm as cm

INSTANCES_DIR = "/home/lucas/Faculdade/AI/trabGA/instances/CVRP/"

def load_vrp(filepath):

    nodes = {}; node_cap = {}; cap = 0; depot = None; state = 0
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line == 'EOF': continue
            if 'CAPACITY' in line and ':' in line:
                cap = int(line.split(':')[1])
            elif 'NODE_COORD_SECTION' in line: state = 1
            elif 'DEMAND_SECTION'    in line: state = 2
            elif 'DEPOT_SECTION'     in line: state = 3
            elif state == 1:
                parts = line.split()
                if len(parts) >= 3:
                    nid, x, y = int(parts[0]), int(parts[1]), int(parts[2])
                    nodes[nid] = (x, y)
            elif state == 2:
                parts = line.split()
                if len(parts) >= 2:
                    nid, demand = int(parts[0]), int(parts[1])
                    node_cap[nid] = demand
            elif state == 3:
                try:
                    nid = int(line)
                    if nid != -1 and depot is None: depot = nid
                except ValueError: pass
    if depot is None:
        depot = next(n for n, d in node_cap.items() if d == 0)
    return nodes, node_cap, cap, depot

def dist(a, b, nodes):

    dx = nodes[a][0] - nodes[b][0]; dy = nodes[a][1] - nodes[b][1]
    return math.sqrt(dx * dx + dy * dy)

def visualize_instance(nodes, node_cap, depot, title="Instância"):
    fig, ax = plt.subplots(figsize=(8, 6))
    for nid, (x, y) in nodes.items():
        if nid == depot:
            ax.scatter(x, y, s=200, color='black', zorder=5, marker='s')
            ax.annotate('Depósito', (x, y), textcoords='offset points', xytext=(5, 5))
        else:
            ax.scatter(x, y, s=60, color='steelblue', zorder=4)
            ax.annotate(str(node_cap[nid]), (x, y), textcoords='offset points',
                        xytext=(3, 3), fontsize=7)
    ax.set_title(title); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

nodes, node_cap, cap, depot = load_vrp(INSTANCES_DIR + "eil33.vrp")
print(f"Instância eil33: {len(nodes)} nós, capacidade={cap}, depósito={depot}")
print(f"Demanda total: {sum(node_cap.values())},  "
      f"veículos mínimos: {math.ceil(sum(node_cap.values()) / cap)}")
visualize_instance(nodes, node_cap, depot, title="eil33 — cidades e demandas")

## 3. Representação dos Indivíduos

Cada **indivíduo** é uma **permutação plana** das cidades (excluindo o depósito):

```
[c3, c7, c12, c2, c15, c8, c5, c11, ...]
```

A divisão em rotas é calculada pela função de avaliação (*decoder greedy*): percorre a permutação e abre uma nova rota quando a próxima cidade excederia a capacidade do veículo.

## 4. Função de Avaliação (Fitness)

1. **Decoder greedy:** converte a permutação em rotas viáveis.
2. **Custo:** retorna `(n_veículos, distância_total)`. Por comparação lexicográfica, minimiza veículos primeiro e distância depois.

```
Permutação: [c3, c7, c12, c2, c15, ...]   cap = 100

c3  (dem 30) → carga: 30
c7  (dem 40) → carga: 70
c12 (dem 40) → 70+40=110 > 100 → fecha rota 1, abre rota 2
c12          → carga: 40
...
```

In [ ]:
def decode_routes(perm, node_cap, cap, depot):

    routes = []; current_route = [depot]; current_load = 0
    for city in perm:
        demand = node_cap[city]
        if current_load + demand <= cap:
            current_route.append(city); current_load += demand
        else:
            current_route.append(depot); routes.append(current_route)
            current_route = [depot, city]; current_load = demand
    current_route.append(depot); routes.append(current_route)
    return routes

def route_distance(route, nodes):

    return sum(dist(route[i], route[i + 1], nodes) for i in range(len(route) - 1))

def evaluate(perm, nodes, node_cap, cap, depot):

    routes = decode_routes(perm, node_cap, cap, depot)
    return (len(routes), sum(route_distance(r, nodes) for r in routes))

def plot_solution(perm, nodes, node_cap, cap, depot, title="Solução"):
    routes = decode_routes(perm, node_cap, cap, depot)
    colors = [cm.tab10(i / max(len(routes), 1)) for i in range(len(routes))]
    fig, ax = plt.subplots(figsize=(9, 7))
    for i, route in enumerate(routes):
        xs = [nodes[c][0] for c in route]; ys = [nodes[c][1] for c in route]
        ax.plot(xs, ys, '-o', color=colors[i], alpha=0.8,
                linewidth=1.5, markersize=5, label=f'Rota {i+1}')
    ax.scatter(*nodes[depot], s=250, color='black', zorder=6, marker='s', label='Depósito')
    n_trucks, total_dist = evaluate(perm, nodes, node_cap, cap, depot)
    ax.set_title(f"{title}\n{n_trucks} veículos, distância total = {total_dist:.1f}")
    ax.legend(loc='upper right', fontsize=7); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

## 5. Inicialização da População

A flag `use_pivot` controla a estratégia:

- `use_pivot=True` *(padrão)*: 1 indivíduo via heurística de pivôs + resto aleatório.
- `use_pivot=False`: toda a população aleatória.

**Heurística de Pivôs:**
1. Calcula $k = \lceil \text{demanda total} / Q \rceil$.
2. Seleciona $k$ pivôs espalhados geograficamente (*farthest-first*).
3. Associa cada cidade ao pivô mais próximo.
4. Ordena cidades em cada grupo por *nearest neighbor* a partir do pivô.
5. Concatena os grupos.

In [ ]:
def select_pivots(n_pivots, cities, nodes, depot):
    pivots = [max(cities, key=lambda c: dist(c, depot, nodes))]
    while len(pivots) < n_pivots:
        remaining = [c for c in cities if c not in pivots]
        nxt = max(remaining, key=lambda c: min(dist(c, p, nodes) for p in pivots))
        pivots.append(nxt)
    return pivots

def nearest_neighbor_order(start, group, nodes):
    remaining = [c for c in group if c != start]
    order = [start]; current = start
    while remaining:
        nearest = min(remaining, key=lambda c: dist(current, c, nodes))
        order.append(nearest); remaining.remove(nearest); current = nearest
    return order

def pivot_individual(cities, nodes, node_cap, cap, depot):
    n_trucks = math.ceil(sum(node_cap[c] for c in cities) / cap)
    pivots   = select_pivots(n_trucks, cities, nodes, depot)
    groups   = {p: [] for p in pivots}
    for city in cities:
        if city in pivots: continue
        nearest = min(pivots, key=lambda p: dist(city, p, nodes))
        groups[nearest].append(city)
    perm = []
    for pivot in pivots:
        perm.extend(nearest_neighbor_order(pivot, groups[pivot], nodes))
    return perm

def initialize_population(pop_size, cities, nodes, node_cap, cap, depot,
                           seed=None, use_pivot=True):

    if seed is not None: random.seed(seed)
    pop = [pivot_individual(cities, nodes, node_cap, cap, depot)] if use_pivot else []
    while len(pop) < pop_size:
        p = cities[:]; random.shuffle(p); pop.append(p)
    return pop

cities_33 = [n for n in nodes if n != depot]
pop_test   = initialize_population(5, cities_33, nodes, node_cap, cap, depot, seed=0)
fit_pivot  = evaluate(pop_test[0], nodes, node_cap, cap, depot)
fit_random = evaluate(pop_test[1], nodes, node_cap, cap, depot)
print(f"Pivô:     {fit_pivot[0]} veículos, distância = {fit_pivot[1]:.1f}")
print(f"Aleatório:{fit_random[0]} veículos, distância = {fit_random[1]:.1f}")
plot_solution(pop_test[0], nodes, node_cap, cap, depot,
              "Solução Inicial — Heurística de Pivôs (eil33)")

## 6. Operadores Genéticos

Implementamos sete operadores em três categorias:

| Categoria | Operadores |
|---|---|
| Seleção (2) | Torneio k=3, Torneio k=7 |
| Cruzamento (3) | OX, PMX, **ER-Q** (7º operador — estado da arte) |
| Mutação (2) | Swap, 2-opt |

### 6.1 Seleção

**Torneio de tamanho k:** sorteia `k` indivíduos aleatoriamente e retorna o de melhor fitness.

| Operador | k | Característica |
|---|---|---|
| `tournament_k3` | 3 | Pressão moderada — mais diversidade |
| `tournament_k7` | 7 | Pressão alta — converge mais rápido |

In [ ]:
def tournament_selection(population, fitnesses, k):
    indices = random.sample(range(len(population)), k)
    best = min(indices, key=lambda i: fitnesses[i])
    return population[best][:]

def tournament_k3(population, fitnesses):

    return tournament_selection(population, fitnesses, 3)

def tournament_k7(population, fitnesses):

    return tournament_selection(population, fitnesses, 7)

### 6.2 Cruzamento (Crossover)

Todos os operadores garantem que cada cidade apareça **exatamente uma vez** no filho.

---

**OX — Order Crossover:**
Copia um segmento do Pai 1 e preenche o restante com as cidades do Pai 2 na ordem em que aparecem.

**PMX — Partially Mapped Crossover:**
Copia um segmento do Pai 1 e usa um mapeamento de posições para resolver conflitos.

---

**ER-Q — Edge Quality Aware Crossover** *(Chitty, Yates e Keedwell — GECCO '22)*

ER-Q é o operador de cruzamento estado da arte para CVRP, baseado no *Edge Recombination* (ER). A ideia central é preservar **arestas dos pais** na solução filha, dando preferência a arestas de **maior qualidade** (menor distância) e respeitando as **restrições de capacidade** do CVRP.

**Algoritmo ER-Q:**
1. Constrói uma **lista de adjacência** $E$: para cada cidade, lista as cidades que aparecem adjacentes a ela em qualquer um dos dois pais.
2. Escolhe uma cidade inicial $N$ aleatoriamente.
3. A cada passo:
   - Adiciona $N$ ao filho.
   - Remove $N$ de todas as listas de adjacência.
   - **Candidatos:** cidades ainda adjacentes a $N$ em $E$ (arestas dos pais). Se não houver nenhuma (*edge failure*), considera todas as cidades restantes.
   - **Probabilidade de seleção** (Equação 2 do artigo):
     $$p_{ij} = \frac{\eta_{ij}^{\alpha} \cdot T_{ij}}{\sum_{l \in N^k} \eta_{il}^{\alpha} \cdot T_{il}}$$
     onde $\eta_{ij} = 1/d_{ij}$ (qualidade da aresta) e $T_{ij}$ é o indicador de tabu:
     $$T_{ij} = \begin{cases} 0 & \text{se demanda}[j] + \text{carga atual} > Q \\ 1 & \text{caso contrário} \end{cases}$$
   - Se todos os candidatos forem tabu (nova rota forçada), reinicia a carga e recomputa sem tabu.

**Diferença chave em relação ao ER padrão:** em vez de simplesmente escolher o vizinho com menos conexões, ER-Q seleciona **probabilisticamente** baseado na distância, e zera a probabilidade de cidades que violariam a capacidade do veículo atual.

In [ ]:
def ox_crossover(p1, p2):

    n = len(p1); a, b = sorted(random.sample(range(n), 2))
    child = [None] * n; child[a:b+1] = p1[a:b+1]
    seg = set(p1[a:b+1]); rem = [x for x in p2 if x not in seg]
    idx = 0
    for i in list(range(b+1, n)) + list(range(0, a)):
        child[i] = rem[idx]; idx += 1
    return child

def pmx_crossover(p1, p2):

    n = len(p1); a, b = sorted(random.sample(range(n), 2))
    child = [None] * n; child[a:b+1] = p1[a:b+1]
    seg = set(p1[a:b+1]); pos_p2 = {v: i for i, v in enumerate(p2)}
    for i in range(a, b+1):
        val = p2[i]
        if val not in seg:
            pos = i
            while child[pos] is not None:
                pos = pos_p2[child[pos]]
            child[pos] = val
    for i in range(n):
        if child[i] is None: child[i] = p2[i]
    return child

def er_q_crossover(p1, p2, nodes, node_cap, cap, alpha=2.0):

    n = len(p1)

    adj = {city: set() for city in p1}
    for perm in [p1, p2]:
        for i in range(n):
            if i > 0:     adj[perm[i]].add(perm[i - 1])
            if i < n - 1: adj[perm[i]].add(perm[i + 1])
    adj = {k: list(v) for k, v in adj.items()}

    N         = random.choice(p1)
    remaining = set(p1)
    result    = []
    current_load = 0

    while remaining:

        if current_load + node_cap[N] > cap:
            current_load = node_cap[N]
        else:
            current_load += node_cap[N]

        result.append(N)
        remaining.discard(N)

        for city in adj:
            if N in adj[city]:
                adj[city].remove(N)

        if not remaining:
            break

        candidates = [c for c in adj[N] if c in remaining]
        if not candidates:
            candidates = list(remaining)

        def quality_scores(cands, load):
            scores = []
            for c in cands:
                if load + node_cap[c] > cap:
                    scores.append(0.0)
                else:
                    scores.append((1.0 / dist(N, c, nodes)) ** alpha)
            return scores

        scores = quality_scores(candidates, current_load)

        if sum(scores) == 0:
            current_load = 0
            scores = [(1.0 / dist(N, c, nodes)) ** alpha for c in candidates]

        total = sum(scores)
        r = random.random() * total
        cumsum = 0
        N = candidates[0]
        for c, s in zip(candidates, scores):
            cumsum += s
            if cumsum >= r:
                N = c
                break

    return result

def make_erq_crossover(nodes, node_cap, cap, alpha=2.0):

    def erq(p1, p2):
        return er_q_crossover(p1, p2, nodes, node_cap, cap, alpha)
    return erq

### 6.3 Mutação

| Operador | Descrição | Intensidade |
|---|---|---|
| **Swap** | Troca duas cidades de posição aleatoriamente | Baixa |
| **2-opt** | Inverte um segmento aleatório da permutação | Média |

In [ ]:
def swap_mutation(perm):

    c = perm[:]; i, j = random.sample(range(len(perm)), 2)
    c[i], c[j] = c[j], c[i]; return c

def two_opt_mutation(perm):

    c = perm[:]; i, j = sorted(random.sample(range(len(perm)), 2))
    c[i:j+1] = c[i:j+1][::-1]; return c

## 7. Loop Principal do Algoritmo Genético

O loop usa **elitismo**: o melhor indivíduo de cada geração é sempre preservado.

A flag `use_pivot` é propagada até a inicialização, permitindo comparar as estratégias nos experimentos.

In [ ]:
def run_ga(nodes, node_cap, cap, depot,
           pop_size=80, n_gen=300,
           p_cross=0.85, p_mut=0.15,
           selection_fn=tournament_k3,
           crossover_fn=ox_crossover,
           mutation_fn=swap_mutation,
           seed=42, use_pivot=True, verbose=False):

    random.seed(seed)
    cities = [n for n in nodes if n != depot]

    population = initialize_population(pop_size, cities, nodes, node_cap, cap, depot,
                                        seed=seed, use_pivot=use_pivot)
    fitnesses = [evaluate(ind, nodes, node_cap, cap, depot) for ind in population]

    best_dist_history = []
    avg_dist_history  = []

    for gen in range(n_gen):
        new_pop = []

        best_idx = min(range(len(population)), key=lambda i: fitnesses[i])
        new_pop.append(population[best_idx][:])

        while len(new_pop) < pop_size:
            p1 = selection_fn(population, fitnesses)
            p2 = selection_fn(population, fitnesses)
            child = crossover_fn(p1, p2) if random.random() < p_cross else p1[:]
            child = mutation_fn(child)   if random.random() < p_mut   else child
            new_pop.append(child)

        population = new_pop
        fitnesses  = [evaluate(ind, nodes, node_cap, cap, depot) for ind in population]

        best_fit = min(fitnesses)
        best_dist_history.append(best_fit[1])
        avg_dist_history.append(sum(f[1] for f in fitnesses) / len(fitnesses))

        if verbose and (gen + 1) % 50 == 0:
            print(f"  Geração {gen+1:4d}: {best_fit[0]} veículos, dist={best_fit[1]:.1f}")

    best_idx = min(range(len(population)), key=lambda i: fitnesses[i])
    return {
        'best_individual'  : population[best_idx],
        'best_fitness'     : fitnesses[best_idx],
        'best_dist_history': best_dist_history,
        'avg_dist_history' : avg_dist_history,
    }

## 8. Experimentos

### Metodologia

| Bloco | O que varia | Fixo | Instância |
|---|---|---|---|
| 1 — Seleção | Torneio k=3 vs k=7 | OX, Swap | eil33 |
| 2 — Cruzamento | OX vs PMX vs **ER-Q** | Torneio k=3, Swap | eil33 |
| 3 — Mutação | Swap vs 2-opt | Torneio k=3, OX | eil33 |
| 4 — Melhor config. | — | melhor dos blocos 1–3 | eil33, eil51, eil76 |
| 5 — Inicialização | `use_pivot=True` vs `False` | melhor config. | eil33, eil51, eil76 |

Cada configuração é executada com **3 sementes** e os resultados são a média entre elas.

In [ ]:
instances = {}
for name in ['eil33', 'eil51', 'eil76']:
    n, nc, c, d = load_vrp(INSTANCES_DIR + f"{name}.vrp")
    instances[name] = {'nodes': n, 'node_cap': nc, 'cap': c, 'depot': d,
                       'cities': [x for x in n if x != d]}
    total = sum(nc.values())
    print(f"{name}: {len(n)} nós, cap={c}, veículos mínimos={math.ceil(total/c)}")

In [ ]:
def run_experiment(inst_name, configs, pop_size=80, n_gen=300,
                   p_cross=0.85, p_mut=0.15,
                   seeds=(0, 1, 2), use_pivot=True):

    inst = instances[inst_name]
    results = {}
    for cfg_name, sel_fn, cross_fn, mut_fn in configs:
        print(f"  [{inst_name}] {cfg_name} ...", end=' ', flush=True)
        all_best = []; all_avg = []; all_fit = []
        for seed in seeds:
            res = run_ga(inst['nodes'], inst['node_cap'], inst['cap'], inst['depot'],
                         pop_size=pop_size, n_gen=n_gen,
                         p_cross=p_cross, p_mut=p_mut,
                         selection_fn=sel_fn, crossover_fn=cross_fn, mutation_fn=mut_fn,
                         seed=seed, use_pivot=use_pivot)
            all_best.append(res['best_dist_history'])
            all_avg.append(res['avg_dist_history'])
            all_fit.append(res['best_fitness'])
        results[cfg_name] = {
            'best_dist_history': [sum(h[g] for h in all_best)/len(seeds) for g in range(n_gen)],
            'avg_dist_history' : [sum(h[g] for h in all_avg) /len(seeds) for g in range(n_gen)],
            'best_fitness'     : min(all_fit),
        }
        bf = results[cfg_name]['best_fitness']
        print(f"veículos={bf[0]}, dist={bf[1]:.1f}")
    return results

def plot_experiment(results, title, ylabel="Distância"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    for name, res in results.items():
        ax1.plot(res['best_dist_history'], label=name)
        ax2.plot(res['avg_dist_history'],  label=name)
    ax1.set_title(f"{title}\nMelhor indivíduo por geração")
    ax2.set_title(f"{title}\nMédia da população por geração")
    for ax in (ax1, ax2):
        ax.set_xlabel("Geração"); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

### Experimento 1 — Operadores de Seleção

Fixamos OX + Swap. Variamos o tamanho do torneio.

In [ ]:
configs_sel = [
    ("Torneio k=3", tournament_k3, ox_crossover, swap_mutation),
    ("Torneio k=7", tournament_k7, ox_crossover, swap_mutation),
]
print("Experimento 1 — Seleção:")
results_sel = run_experiment('eil33', configs_sel)
plot_experiment(results_sel, "Experimento 1 — Seleção (eil33)")

### Experimento 2 — Operadores de Cruzamento

Fixamos Torneio k=3 + Swap. Comparamos OX, PMX e ER-Q.

> **Nota:** ER-Q é criado via `make_erq_crossover`, pois precisa das coordenadas e capacidades da instância.

In [ ]:
inst_33 = instances['eil33']
erq_33  = make_erq_crossover(inst_33['nodes'], inst_33['node_cap'], inst_33['cap'])

configs_cross = [
    ("OX",   tournament_k3, ox_crossover,  swap_mutation),
    ("PMX",  tournament_k3, pmx_crossover, swap_mutation),
    ("ER-Q", tournament_k3, erq_33,        swap_mutation),
]
print("Experimento 2 — Cruzamento:")
results_cross = run_experiment('eil33', configs_cross)
plot_experiment(results_cross, "Experimento 2 — Cruzamento (eil33)")

### Experimento 3 — Operadores de Mutação

Fixamos Torneio k=3 + OX. Variamos a mutação.

In [ ]:
configs_mut = [
    ("Swap",  tournament_k3, ox_crossover, swap_mutation),
    ("2-opt", tournament_k3, ox_crossover, two_opt_mutation),
]
print("Experimento 3 — Mutação:")
results_mut = run_experiment('eil33', configs_mut)
plot_experiment(results_mut, "Experimento 3 — Mutação (eil33)")

### Experimento 4 — Melhor Configuração em Todas as Instâncias

Com base nos resultados anteriores, aplicamos a melhor combinação nas três instâncias.

> Ajuste `best_selection`, `best_crossover_name` e `best_mutation` conforme os resultados dos Experimentos 1–3.
> O ER-Q é recriado para cada instância via `make_erq_crossover`.

In [ ]:
best_selection       = tournament_k3
best_crossover_name  = "ER-Q"
best_mutation        = swap_mutation

print("Experimento 4 — Melhor configuração em todas as instâncias:")
results_all = {}
for inst_name in ['eil33', 'eil51', 'eil76']:
    inst     = instances[inst_name]
    erq_fn   = make_erq_crossover(inst['nodes'], inst['node_cap'], inst['cap'])
    cross_map = {"OX": ox_crossover, "PMX": pmx_crossover, "ER-Q": erq_fn}
    best_crossover = cross_map[best_crossover_name]
    configs = [("Melhor config.", best_selection, best_crossover, best_mutation)]
    results_all[inst_name] = run_experiment(inst_name, configs)

In [ ]:
for inst_name in ['eil33', 'eil51', 'eil76']:
    plot_experiment(results_all[inst_name],
                    title=f"Experimento 4 — {inst_name}", ylabel="Distância total")

In [ ]:
for inst_name in ['eil33', 'eil51', 'eil76']:
    inst = instances[inst_name]
    erq_fn = make_erq_crossover(inst['nodes'], inst['node_cap'], inst['cap'])
    cross_map = {"OX": ox_crossover, "PMX": pmx_crossover, "ER-Q": erq_fn}
    best_res = run_ga(inst['nodes'], inst['node_cap'], inst['cap'], inst['depot'],
                      pop_size=80, n_gen=300, p_cross=0.85, p_mut=0.15,
                      selection_fn=best_selection,
                      crossover_fn=cross_map[best_crossover_name],
                      mutation_fn=best_mutation, seed=0, use_pivot=True)
    fit = best_res['best_fitness']
    print(f"{inst_name}: {fit[0]} veículos, distância = {fit[1]:.1f}")
    plot_solution(best_res['best_individual'], inst['nodes'], inst['node_cap'],
                  inst['cap'], inst['depot'], title=f"Melhor solução — {inst_name}")

### Experimento 5 — Impacto da Heurística de Pivôs

Compara `use_pivot=True` (1 indivíduo via heurística de pivôs + resto aleatório) contra `use_pivot=False` (população totalmente aleatória).

In [ ]:
print("Experimento 5 — Com pivô vs. sem pivô:")
results_pivot = {}
for inst_name in ['eil33', 'eil51', 'eil76']:
    inst   = instances[inst_name]
    erq_fn = make_erq_crossover(inst['nodes'], inst['node_cap'], inst['cap'])
    cross_map = {"OX": ox_crossover, "PMX": pmx_crossover, "ER-Q": erq_fn}
    best_cross = cross_map[best_crossover_name]
    cfg = [("Melhor config.", best_selection, best_cross, best_mutation)]

    print(f"\n  {inst_name} — use_pivot=True:")
    r_with    = run_experiment(inst_name, cfg, use_pivot=True)
    print(f"  {inst_name} — use_pivot=False:")
    r_without = run_experiment(inst_name, cfg, use_pivot=False)

    results_pivot[inst_name] = {
        "Com pivô" : r_with["Melhor config."],
        "Sem pivô" : r_without["Melhor config."],
    }

In [ ]:
for inst_name in ['eil33', 'eil51', 'eil76']:
    plot_experiment(results_pivot[inst_name],
                    title=f"Experimento 5 — Inicialização ({inst_name})",
                    ylabel="Distância total")

## 9. Resultados

*(Preencher após executar os experimentos.)*

### Tabela Resumo — Experimento 4

| Instância | Veículos | Distância Total | Configuração |
|---|---|---|---|
| eil33 | — | — | — |
| eil51 | — | — | — |
| eil76 | — | — | — |

### Análise dos Operadores

- **Seleção:** *(qual torneio performou melhor e por quê)*
- **Cruzamento:** *(comparar OX, PMX e ER-Q — o ER-Q tende a ser superior por preservar arestas de qualidade e respeitar a capacidade durante a construção)*
- **Mutação:** *(comparar Swap e 2-opt)*

### Impacto da Heurística de Pivôs

*(Descrever se a inicialização com pivôs acelerou a convergência ou melhorou o resultado final.)*

## 10. Conclusões

*(Preencher após a análise dos resultados.)*

O trabalho implementou um Algoritmo Genético completo para o CVRP com sete operadores genéticos. O operador ER-Q destaca-se por considerar a qualidade das arestas durante a construção do filho, penalizando movimentos que violam a capacidade dos veículos — abordagem diretamente alinhada com a estrutura do problema CVRP.